In [1]:
import tensorflow as tf 
import pandas as pd

PATH = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
PATH_test = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"
COLUMNS = ['age','workclass','fnlwgt','education','education_num','marital','occupation','relationship','race','sex','capital_gain','capital_loss','hours_week','native_country','label']


In [2]:
df_train = pd.read_csv( PATH, skipinitialspace=True, names= COLUMNS, index_col=False)
df_test = pd.read_csv( PATH_test,skiprows=1, skipinitialspace=True, names= COLUMNS, index_col=False)

print(df_train.shape, df_test.shape)

print(df_train.dtypes)

(32561, 15) (16281, 15)
age               int64
workclass           str
fnlwgt            int64
education           str
education_num     int64
marital             str
occupation          str
relationship        str
race                str
sex                 str
capital_gain      int64
capital_loss      int64
hours_week        int64
native_country      str
label               str
dtype: object


In [3]:
label = {'<=50K': 0, '>50K': 1}
df_train.label = [label[item] for item in df_train.label]
label_t = {'<=50K.': 0, '>50K.': 1}
df_test.label = [label_t[item] for item in df_test.label]

In [4]:
print(df_train["label"].value_counts())
print(df_test["label"].value_counts())

print(df_train.dtypes)

label
0    24720
1     7841
Name: count, dtype: int64
label
0    12435
1     3846
Name: count, dtype: int64
age               int64
workclass           str
fnlwgt            int64
education           str
education_num     int64
marital             str
occupation          str
relationship        str
race                str
sex                 str
capital_gain      int64
capital_loss      int64
hours_week        int64
native_country      str
label             int64
dtype: object


In [5]:
# Add features to the bucket
# Define continuous list

CONT_FEATURES = ['age', 'fnlwgt', 'capital_gain', 'education_num', 'capital_loss', 'hours_week']
# Define the categorial list
CATE_FEATURES = ['workclass', 'education', 'marital', 'occupation', 'relationship', 'race', 'sex', 'native_country']

from pandas.core.util.hashing import hash_pandas_object
# from sqlalchemy.orm import relationship
continuous_features = [tf.feature_column.numeric_column(k) for k in CONT_FEATURES]
relationship = tf.feature_column.categorical_column_with_vocabulary_list('relationship',['Husband','Not-in-family', 'Wife', 'Own-child', 'Unmarried','Other-relative'])
categorical_features = [tf.feature_column.categorical_column_with_hash_bucket(k,hash_bucket_size = 1000) for k in CATE_FEATURES]


Instructions for updating:
Use Keras preprocessing layers instead, either directly or via the `tf.keras.utils.FeatureSpace` utility. Each of `tf.feature_column.*` has a functional equivalent in `tf.keras.layers` for feature preprocessing when training a Keras model.


Instructions for updating:
Use Keras preprocessing layers instead, either directly or via the `tf.keras.utils.FeatureSpace` utility. Each of `tf.feature_column.*` has a functional equivalent in `tf.keras.layers` for feature preprocessing when training a Keras model.


Instructions for updating:
Use Keras preprocessing layers instead, either directly or via the `tf.keras.utils.FeatureSpace` utility. Each of `tf.feature_column.*` has a functional equivalent in `tf.keras.layers` for feature preprocessing when training a Keras model.


In [6]:
import os
model_dir = os.path.abspath('ongoing/train')
model = tf.estimator.LinearClassifier(n_classes=2, model_dir=model_dir, feature_columns=categorical_features + continuous_features)


Instructions for updating:
Use tf.keras instead.


Instructions for updating:
Use tf.keras instead.


Instructions for updating:
Use tf.keras instead.


Instructions for updating:
Use tf.keras instead.


INFO:tensorflow:Using default config.


INFO:tensorflow:Using config: {'_model_dir': 'E:\\LANGCHAIN\\Deeplearning\\ongoing\\train', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}


In [7]:
import numpy as np

# Prepare data types required by TensorFlow Estimators
for col in CATE_FEATURES:
    df_train[col] = df_train[col].astype('object')
    df_test[col] = df_test[col].astype('object')

for col in CONT_FEATURES:
    df_train[col] = df_train[col].astype(np.float32)
    df_test[col] = df_test[col].astype(np.float32)

# Input function using tf.data.Dataset for high performance and compatibility
def get_input_fn(df, num_epochs=None, batch_size=128, shuffle=True):
    def input_fn():
        features = {col: df[col].to_numpy() for col in CONT_FEATURES + CATE_FEATURES}
        labels = df['label'].to_numpy(dtype=np.int32)
        dataset = tf.data.Dataset.from_tensor_slices((features, labels))
        if shuffle:
            dataset = dataset.shuffle(buffer_size=len(df))
        if num_epochs:
            dataset = dataset.repeat(num_epochs)
        else:
            dataset = dataset.repeat()
        dataset = dataset.batch(batch_size)
        return dataset
    return input_fn


In [8]:
# Train the LinearClassifier Estimator
model.train(input_fn=get_input_fn(df_train, num_epochs=None, batch_size=128, shuffle=True), steps=500)


Instructions for updating:
Use tf.keras instead.


INFO:tensorflow:Calling model_fn.


Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


Instructions for updating:
Use tf.keras instead.


INFO:tensorflow:Done calling model_fn.


...

INFO:tensorflow:loss = 197.00954, step = 1201 (0.090 sec)


INFO:tensorflow:Calling checkpoint listeners before saving checkpoint 1301...


INFO:tensorflow:Saving checkpoints for 1301 into E:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt.


INFO:tensorflow:Calling checkpoint listeners after saving checkpoint 1301...


INFO:tensorflow:Loss for final step: 67.53245.


In [9]:
# Evaluate on the test dataset
results = model.evaluate(input_fn=get_input_fn(df_test, num_epochs=1, batch_size=128, shuffle=False))
print('Test Accuracy:', results['accuracy'])


INFO:tensorflow:Calling model_fn.


INFO:tensorflow:Done calling model_fn.


INFO:tensorflow:Starting evaluation at 2026-09-24T19:41:19


Instructions for updating:
Use tf.keras instead.


INFO:tensorflow:Graph was finalized.


...

INFO:tensorflow:Inference Time : 1.01239s


INFO:tensorflow:Finished evaluation at 2026-09-24-19:41:20


INFO:tensorflow:Saving dict for global step 1301: accuracy = 0.2371476, accuracy_baseline = 0.76377374, auc = 0.50265384, auc_precision_recall = 0.23718779, average_loss = 66.62722, global_step = 1301, label/mean = 0.23622628, loss = 66.745186, precision = 0.23644412, prediction/mean = 0.99886066, recall = 1.0


INFO:tensorflow:Saving 'checkpoint_path' summary for global step 1301: E:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-1301


Test Accuracy: 0.2371476
